[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week1_data_foundations/day05_missing_data/day05_notebook.ipynb)

# Day 5 / 42: Missing Data — MCAR, MAR, MNAR
**#42DaysOfML**

---

## What You Will Learn
- The 3 types of missing data and why they require different strategies
- How to detect which type you are dealing with
- Mean, median, KNN, and MICE (IterativeImputer) strategies
- Missingness indicator encoding — and why it outperforms all other strategies on MNAR data
- The production bias that breaks healthcare models
- A head-to-head model performance comparison: 4 strategies on the same dataset

**Dataset:** Synthetic clinical readmission dataset (realistic missingness patterns)

**Time to complete:** 50–70 minutes

---

## Step 0: Install and Import

In [ ]:
# Run on Google Colab — missingno is not pre-installed
!pip install missingno -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer   # must import before IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression, BayesianRidge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
np.random.seed(42)

print("All libraries loaded successfully")

## Step 1: The 3 Types of Missing Data

This is the most important thing to understand before writing a single line of imputation code.
The type of missingness determines which strategy is valid.
Using the wrong strategy does not just underperform — it introduces **systematic bias** into your model.

---

### MCAR — Missing Completely At Random
The probability of a value being missing has **no relationship** to any variable, observed or unobserved.  
Example: A survey where some forms were randomly lost in transit.

**Safe to:** drop rows or use simple imputation (mean/median). Bias risk is low.

---

### MAR — Missing At Random
The probability of missingness **is related to other observed variables**, but not to the value that is missing itself.  
Example: Men are less likely to report their weight. You can predict who is likely missing from gender, but the weight value itself doesn't drive the missingness.

**Safe to:** use imputation strategies that condition on other observed features (KNN, MICE). Dropping rows introduces selection bias.

---

### MNAR — Missing Not At Random
The probability of missingness **is related to the value that is missing**, even after conditioning on other variables.  
Example: Patients with very high blood pressure skip their measurement. The measurement is missing precisely because it is extreme.

**Dangerous:** Any imputation replaces the most extreme values with average ones, systematically pulling your model away from the cases it most needs to handle correctly. Add a **missingness indicator column** to preserve this signal.

## Step 2: Build a Realistic Dataset With All 3 Types

We build a clinical readmission dataset where:
- `age` has **MCAR** missingness (random data entry gaps)
- `bmi` has **MAR** missingness (higher-income patients less likely to report weight)
- `systolic_bp` has **MNAR** missingness (sicker patients skip their BP measurement)

In [ ]:
np.random.seed(42)
n = 500

# Ground truth values — what the data ACTUALLY is
age           = np.random.normal(45, 15, n).clip(18, 90)
systolic_bp   = 120 + 0.4 * age + np.random.normal(0, 12, n)  # BP increases with age
bmi           = np.random.normal(26, 5, n).clip(15, 50)
income        = np.random.exponential(45000, n).clip(10000, 200000)
health_score  = systolic_bp - 120  # proxy for illness severity

# ── MCAR: random 15% of age values go missing ─────────────────────
mcar_mask    = np.random.random(n) < 0.15
age_observed = age.copy().astype(float)
age_observed[mcar_mask] = np.nan

# ── MAR: higher income → higher probability of skipping weight entry
mar_prob       = 1 / (1 + np.exp(-(income - 50000) / 15000)) * 0.4
mar_mask       = np.random.random(n) < mar_prob
bmi_observed   = bmi.copy().astype(float)
bmi_observed[mar_mask] = np.nan

# ── MNAR: sicker patients (high BP) more likely to skip measurement
mnar_prob      = 1 / (1 + np.exp(-(health_score - 15) / 5)) * 0.5
mnar_mask      = np.random.random(n) < mnar_prob
bp_observed    = systolic_bp.copy().astype(float)
bp_observed[mnar_mask] = np.nan

df = pd.DataFrame({
    'age':          age_observed,
    'bmi':          bmi_observed,
    'systolic_bp':  bp_observed,
    'income':       income,
    'readmitted':   (health_score > 20).astype(int)
})

print("Dataset shape:", df.shape)
print()
print("Missing value summary:")
missing_report = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %':     (df.isnull().sum() / len(df) * 100).round(1),
    'Type':          ['MCAR', 'MAR', 'MNAR', 'None', 'None']
})
print(missing_report)
print()
print("Target distribution:")
print(df['readmitted'].value_counts())
print(f"Readmission rate: {df['readmitted'].mean():.1%}")

## Step 3: Visualise the Missing Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Matrix: white = missing, dark = present
# Look for vertical white streaks (columns missing together)
msno.matrix(df, ax=axes[0], sparkline=False, fontsize=10, color=(0.2, 0.4, 0.7))
axes[0].set_title('Missing Value Matrix\n(white stripe = missing record)', fontsize=12, fontweight='bold')

# Bar: overall completeness per column
msno.bar(df, ax=axes[1], fontsize=10, color='steelblue')
axes[1].set_title('Column Completeness (%)', fontsize=12, fontweight='bold')

plt.suptitle('Step 1: Where Is Data Missing?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Reading the matrix:")
print("  - age and bmi have scattered white stripes → independent missingness")
print("  - systolic_bp has the most white stripes → highest missingness rate")
print("  - No obvious vertical alignment between columns → they're not missing together")

## Step 4: Detect the Missingness Type

The core diagnostic tool: correlate each column's **missingness indicator** (1 = missing, 0 = present)
with all other observed variables.

- **Low correlations everywhere** → MCAR
- **High correlation with another observed column** → MAR
- **High correlation with the target or outcome** → MNAR (the hardest case)

In [ ]:
# Build a missingness indicator dataframe
missing_indicators = df.isnull().astype(int).add_suffix('_missing')

# Only look at columns that actually have missingness
cols_with_missing = ['age', 'bmi', 'systolic_bp']

# Combine indicators with observed columns for correlation
diag_df = pd.concat([
    df[['income', 'readmitted']],
    missing_indicators[['age_missing', 'bmi_missing', 'systolic_bp_missing']]
], axis=1)

# Fill NaN in systolic_bp with median for correlation calc
diag_df['systolic_bp_approx'] = df['systolic_bp'].fillna(df['systolic_bp'].median())

print("=" * 65)
print("MISSINGNESS TYPE DIAGNOSIS")
print("=" * 65)

for col in cols_with_missing:
    ind = f'{col}_missing'
    corr_income  = diag_df[ind].corr(diag_df['income'])
    corr_readmit = diag_df[ind].corr(diag_df['readmitted'].astype(float))
    corr_bp      = diag_df[ind].corr(diag_df['systolic_bp_approx'])

    print(f"\n{col.upper()} missingness correlations:")
    print(f"  with income       : {corr_income:+.3f}")
    print(f"  with readmitted   : {corr_readmit:+.3f}")
    print(f"  with systolic_bp  : {corr_bp:+.3f}")

    max_abs = max(abs(corr_income), abs(corr_readmit), abs(corr_bp))
    if max_abs < 0.1:
        verdict = "MCAR — no strong correlation with any variable"
    elif abs(corr_income) > abs(corr_readmit):
        verdict = "MAR  — missingness predicted by an observed variable (income)"
    else:
        verdict = "MNAR — missingness correlated with outcome/severity"
    print(f"  Verdict: {verdict}")

print()
print("Note: In production, use Little's MCAR test for a formal statistical test.")
print("This correlation approach is a fast, practical proxy that works well in most cases.")

In [ ]:
# Visualise why each column has its missingness type
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MCAR — age: missingness shows no pattern with income
age_miss_ind = df['age'].isnull().astype(int)
jitter = np.random.normal(0, 0.015, n)
axes[0].scatter(df['income'], age_miss_ind + jitter,
                c=age_miss_ind, cmap='RdYlGn_r', alpha=0.4, s=18)
axes[0].set_xlabel('Income')
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(['Present', 'Missing'])
axes[0].set_title('AGE (MCAR)\nNo pattern with income', fontsize=11, fontweight='bold')

# MAR — bmi: higher income → more likely missing
bmi_miss_ind = df['bmi'].isnull().astype(int)
axes[1].scatter(df['income'], bmi_miss_ind + jitter,
                c=bmi_miss_ind, cmap='RdYlGn_r', alpha=0.4, s=18)
axes[1].set_xlabel('Income')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Present', 'Missing'])
axes[1].set_title('BMI (MAR)\nHigh income → more missing', fontsize=11, fontweight='bold')

# MNAR — systolic_bp: sicker patients (high true BP) skip measurement
bp_miss_ind = df['systolic_bp'].isnull().astype(int)
present_bp = systolic_bp[~bp_miss_ind.astype(bool)]
missing_bp = systolic_bp[bp_miss_ind.astype(bool)]
axes[2].boxplot([present_bp, missing_bp],
                labels=['BP Recorded', 'BP Missing'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[2].set_ylabel('True Systolic BP (mmHg)')
axes[2].set_title('SYSTOLIC BP (MNAR)\nMissing patients have higher true BP',
                  fontsize=11, fontweight='bold')

plt.suptitle('Why Each Column Has Its Missingness Type', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"True mean BP (recorded patients): {present_bp.mean():.1f} mmHg")
print(f"True mean BP (missing patients) : {missing_bp.mean():.1f} mmHg")
print(f"Gap: {missing_bp.mean() - present_bp.mean():.1f} mmHg")
print("→ Missing patients are systematically sicker. Mean imputation will hide this.")

## Step 5: Imputation Strategies

Now we apply each strategy and compare what happens to the data distribution and model performance.

In [ ]:
# Strategy 1 & 2: Mean and Median Imputation
# Fast, simple, but ignores relationships between columns
# ONLY appropriate for MCAR data

df_mean   = df.copy()
df_median = df.copy()

for col in ['age', 'bmi', 'systolic_bp']:
    df_mean[col].fillna(df_mean[col].mean(), inplace=True)
    df_median[col].fillna(df_median[col].median(), inplace=True)

print("Mean imputation — column stats after:")
print(df_mean[['age', 'bmi', 'systolic_bp']].describe().round(2))

print("\nMedian imputation — column stats after:")
print(df_median[['age', 'bmi', 'systolic_bp']].describe().round(2))

print()
print("What to notice:")
print("  mean()  and median()  are near the center of each column")
print("  std is reduced after mean/median imputation — distribution is narrowed")
print("  For MCAR data (age), this is acceptable")
print("  For MNAR data (systolic_bp), the extreme values being filled with 'average'")
print("  hides exactly the signal the model needs — the sickest patients")

In [ ]:
# Strategy 3: KNN Imputation
# Imputes each missing value using the average of k nearest neighbours
# in the feature space (measured across non-missing columns)
# Better than mean/median for MAR because it conditions on other features

# IMPORTANT: fit on training data only in production. Here we fit on full df for demo.
knn_imputer = KNNImputer(n_neighbors=5)
feature_cols = ['age', 'bmi', 'systolic_bp', 'income']

df_knn = df.copy()
df_knn[feature_cols] = knn_imputer.fit_transform(df_knn[feature_cols])

print("KNN imputation (k=5) — column stats after:")
print(df_knn[['age', 'bmi', 'systolic_bp']].describe().round(2))

# Visualise: KNN better preserves the distribution shape vs mean
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
columns_to_check = ['age', 'bmi', 'systolic_bp']
titles = ['AGE (MCAR)', 'BMI (MAR)', 'SYSTOLIC BP (MNAR)']

for i, (col, title) in enumerate(zip(columns_to_check, titles)):
    true_vals  = {'age': age, 'bmi': bmi, 'systolic_bp': systolic_bp}[col]
    axes[i].hist(true_vals,        bins=25, alpha=0.5, label='True',   color='green',    density=True)
    axes[i].hist(df_mean[col],     bins=25, alpha=0.4, label='Mean',   color='red',      density=True)
    axes[i].hist(df_knn[col],      bins=25, alpha=0.4, label='KNN',    color='steelblue',density=True)
    axes[i].set_title(title, fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=9)

plt.suptitle('Distribution Shape: True vs Mean vs KNN Imputation', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print("Key observation:")
print("  KNN preserves the distribution shape better than mean imputation")
print("  For MNAR (systolic_bp): both strategies still fail to capture the extreme-value signal")
print("  → That's what the missingness indicator handles")

In [ ]:
# Strategy 4: Missingness Indicator Encoding
# For MNAR columns: the fact that a value is MISSING is itself a feature
# This is the strategy LinkedIn uses for engagement data
# (non-engagement is informative — the absence of a click tells you something)

df_indicator = df.copy()

for col in ['age', 'bmi', 'systolic_bp']:
    # Step 1: Add a binary column capturing the fact of missingness
    df_indicator[f'{col}_was_missing'] = df_indicator[col].isnull().astype(int)
    # Step 2: Impute the original column so it has no NaNs
    df_indicator[col].fillna(df_indicator[col].median(), inplace=True)

print("Columns after missingness indicator encoding:")
print(df_indicator.columns.tolist())
print()
print("Missingness indicator value counts:")
for col in ['age', 'bmi', 'systolic_bp']:
    ind_col = f'{col}_was_missing'
    print(f"  {ind_col}: {df_indicator[ind_col].sum()} patients flagged ({df_indicator[ind_col].mean():.1%})")

print()
print("Why this works for MNAR:")
print("  The systolic_bp_was_missing column captures which patients skipped their measurement")
print("  The model can learn: 'patients where BP was missing have higher readmission risk'")
print("  Without the indicator, the model has no way to know this group exists")

In [ ]:
# Strategy 5: MICE — Multiple Imputation by Chained Equations
# sklearn's IterativeImputer implements MICE
# Imputes each column iteratively using all other columns as predictors
# Best for MAR data when columns are correlated
# Computationally heavier than KNN — use for important features only

mice_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,
    random_state=42
)

df_mice = df.copy()
df_mice[feature_cols] = mice_imputer.fit_transform(df_mice[feature_cols])

print("MICE (IterativeImputer) — column stats after:")
print(df_mice[['age', 'bmi', 'systolic_bp']].describe().round(2))

print()
print("MICE uses all other columns as predictors for each imputed column.")
print("For our dataset: imputing BMI uses age, systolic_bp, and income together.")
print("This is why it performs better than single-column imputation on correlated features.")
print()
print("Production note: In sklearn pipelines, MICE can cause data leakage if not fitted")
print("on training data only. Always fit the imputer on train, transform on test.")

## Step 6: Head-to-Head Model Performance Comparison

Same model, same dataset, same target — only the imputation strategy changes.  
This shows exactly what is at stake when you choose the wrong strategy.

In [ ]:
target = 'readmitted'
base_features = ['age', 'bmi', 'systolic_bp', 'income']
indicator_features = base_features + ['age_was_missing', 'bmi_was_missing', 'systolic_bp_was_missing']

strategies = {
    'Mean':      (df_mean,      base_features),
    'Median':    (df_median,    base_features),
    'KNN':       (df_knn,       base_features),
    'MICE':      (df_mice,      base_features),
    'Indicator': (df_indicator, indicator_features),
}

results = {}
scaler = StandardScaler()
model  = LogisticRegression(max_iter=500, random_state=42)

print(f"{'Strategy':<12}  {'AUC-ROC':>8}  {'±':>6}")
print("-" * 32)

for name, (df_imp, feat_cols) in strategies.items():
    X = scaler.fit_transform(df_imp[feat_cols])
    y = df_imp[target]
    scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
    results[name] = (scores.mean(), scores.std())
    print(f"{name:<12}  {scores.mean():>8.4f}  {scores.std():>6.4f}")

best = max(results, key=lambda k: results[k][0])
worst = min(results, key=lambda k: results[k][0])
gap = results[best][0] - results[worst][0]

print()
print(f"Best strategy : {best} (AUC {results[best][0]:.4f})")
print(f"Worst strategy: {worst} (AUC {results[worst][0]:.4f})")
print(f"Gap           : {gap:.4f} ({gap*100:.1f} AUC points)")
print()
print("For a clinical model, this gap can mean the difference between")
print("a model that correctly flags high-risk patients and one that misses them.")

In [ ]:
# Visualise the comparison
names  = list(results.keys())
aucs   = [results[n][0] for n in names]
errors = [results[n][1] for n in names]
colors = ['#E53935', '#FB8C00', '#1E88E5', '#8E24AA', '#43A047']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, aucs, color=colors, edgecolor='white', width=0.55, yerr=errors, capsize=5)
ax.set_ylim(0.75, 1.00)
ax.set_ylabel('AUC-ROC (5-fold CV)', fontsize=12)
ax.set_title('Imputation Strategy Comparison: Same Model, Same Data, Different Strategy',
             fontsize=13, fontweight='bold')
ax.axhline(y=aucs[0], color='gray', linestyle='--', alpha=0.4, label=f'Mean baseline ({aucs[0]:.3f})')
ax.legend(fontsize=10)

for bar, auc in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{auc:.4f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

indicator_lift = results['Indicator'][0] - results['Mean'][0]
print(f"\nMissingness indicator vs mean imputation: +{indicator_lift*100:.1f} AUC points")
print("That lift comes entirely from letting the model learn from the absence of data.")
print("No new features. No new algorithm. Just treating missingness as a signal.")

## Step 7: The Production Problem — Healthcare Model Bias

This is one of the most cited failure patterns in clinical ML.  
Understanding it protects you from repeating it.

In [ ]:
# The healthcare model bias scenario
# A team builds a readmission prediction model
# systolic_bp has 30% missingness — they impute with the column mean
# The model trains without error and reports acceptable performance
# In deployment, it consistently under-flags the sickest patients

print("=" * 60)
print("THE MNAR BIAS PROBLEM IN PRODUCTION")
print("=" * 60)

# True statistics
true_mean_bp        = systolic_bp.mean()
observed_mean_bp    = systolic_bp[~mnar_mask].mean()
missing_mean_bp     = systolic_bp[mnar_mask].mean()

# Readmission rate by group
readmitted_arr      = (health_score > 20).astype(int)
readmit_observed    = readmitted_arr[~mnar_mask].mean()
readmit_missing_bp  = readmitted_arr[mnar_mask].mean()

print(f"\nBlood pressure statistics:")
print(f"  True mean BP (all patients)     : {true_mean_bp:.1f} mmHg")
print(f"  Observed mean BP (recorded)     : {observed_mean_bp:.1f} mmHg")
print(f"  True mean BP (missing patients) : {missing_mean_bp:.1f} mmHg")
print(f"  Gap                             : +{missing_mean_bp - observed_mean_bp:.1f} mmHg")

print(f"\nReadmission rates:")
print(f"  Patients with recorded BP       : {readmit_observed:.1%}")
print(f"  Patients with MISSING BP        : {readmit_missing_bp:.1%}")
print(f"  Gap                             : +{(readmit_missing_bp - readmit_observed)*100:.1f} percentage points")

print(f"""
What happens with mean imputation:
  - {mnar_mask.sum()} patients had no BP recorded
  - Their true BP averaged {missing_mean_bp:.1f} mmHg (higher = sicker)
  - Mean imputation replaces all {mnar_mask.sum()} with {observed_mean_bp:.1f} mmHg
  - The model now sees these patients as average-health, not high-risk
  - Readmission rate for this group: {readmit_missing_bp:.1%} — but the model treats them as low-risk

The fix:
  - Add systolic_bp_was_missing = 1 for these patients
  - The model learns to use the indicator as a high-risk signal
  - This is documented in published NHS and academic clinical ML literature
  as one of the most common avoidable errors in healthcare prediction models
""")

## Step 8: Decision Guide — Which Strategy to Use

This is the framework you apply on every new column with missing data.

In [ ]:
def missing_data_strategy(col_name, pct_missing, missingness_type, is_important_feature, skewed=False):
    """
    Returns the recommended imputation strategy based on missingness type
    and column characteristics.

    Parameters
    ----------
    col_name            : str   — column name
    pct_missing         : float — percentage of values missing (0-100)
    missingness_type    : str   — 'MCAR', 'MAR', or 'MNAR'
    is_important_feature: bool  — is this column correlated with the target?
    skewed              : bool  — is the column distribution skewed?
    """
    print(f"\nColumn: {col_name}  |  Missing: {pct_missing:.1f}%  |  Type: {missingness_type}")
    print("-" * 60)

    if pct_missing > 60:
        print("  WARNING: >60% missing. Consider dropping this column entirely.")
        print("  If domain context says it's critical, model the missingness separately.")
        return

    if missingness_type == 'MCAR':
        if pct_missing < 5:
            print("  Strategy: Drop missing rows (small loss, no bias)")
        elif skewed:
            print("  Strategy: Median imputation (skewed distribution — median is more robust)")
        else:
            print("  Strategy: Mean imputation (acceptable for MCAR)")

    elif missingness_type == 'MAR':
        if is_important_feature:
            print("  Strategy: KNN or MICE imputation (conditions on other features)")
            print("  Also add missingness indicator as a safety net")
        else:
            print("  Strategy: Median imputation is acceptable if feature importance is low")

    elif missingness_type == 'MNAR':
        print("  Strategy: ALWAYS add missingness indicator column")
        print("  Impute original column with median (the imputed value matters less here)")
        print("  The indicator column carries the real signal")
        if is_important_feature:
            print("  Also try MICE imputation on the original column for completeness")

    else:
        print("  Unknown type. Default: median imputation + missingness indicator")


# Apply to our three columns
missing_data_strategy('age',          pct_missing=15.8, missingness_type='MCAR',
                      is_important_feature=True,  skewed=False)

missing_data_strategy('bmi',          pct_missing=15.8, missingness_type='MAR',
                      is_important_feature=True,  skewed=False)

missing_data_strategy('systolic_bp',  pct_missing=31.0, missingness_type='MNAR',
                      is_important_feature=True,  skewed=False)

## Step 9: Production Pipeline — Imputer Fitted on Train, Transformed on Test

This is the most common data leakage mistake with imputation.  
**Never fit the imputer on the full dataset. Fit on train only.**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Using df_indicator (median + indicator columns) as our best strategy
X = df_indicator[indicator_features]
y = df_indicator[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# WRONG WAY — fitting imputer on all data before splitting
# (This leaks test statistics into training)
print("WRONG WAY (data leakage):")
print("  imputer.fit_transform(df_full)  # sees test data during fit — LEAKAGE")
print("  X_train, X_test = train_test_split(...)")
print()

# RIGHT WAY — imputer fit only on training data
print("RIGHT WAY (no leakage):")

# Since we already handled missingness manually above,
# demonstrate with a fresh copy that still has NaNs
X_raw = df[base_features]   # has NaNs
y_raw = df[target]

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# Fit on TRAIN only
knn_prod = KNNImputer(n_neighbors=5)
knn_prod.fit(X_train_raw)                           # fit on train
X_train_imputed = knn_prod.transform(X_train_raw)   # transform train
X_test_imputed  = knn_prod.transform(X_test_raw)    # transform test (no fitting!)

scaler_prod = StandardScaler()
scaler_prod.fit(X_train_imputed)
X_train_scaled = scaler_prod.transform(X_train_imputed)
X_test_scaled  = scaler_prod.transform(X_test_imputed)

model_prod = LogisticRegression(max_iter=500, random_state=42)
model_prod.fit(X_train_scaled, y_train_raw)

from sklearn.metrics import roc_auc_score
train_auc = roc_auc_score(y_train_raw, model_prod.predict_proba(X_train_scaled)[:,1])
test_auc  = roc_auc_score(y_test_raw,  model_prod.predict_proba(X_test_scaled)[:,1])

print(f"  Train AUC: {train_auc:.4f}")
print(f"  Test AUC : {test_auc:.4f}")
print(f"  Gap      : {train_auc - test_auc:.4f}  (small gap = good generalisation)")
print()
print("In production: save the fitted knn_prod and scaler_prod objects.")
print("When new data arrives, call .transform() — never .fit_transform().")
print("The imputer must use the same statistics it learned from the training distribution.")

## Practice Exercise

We use the Titanic dataset. Your tasks:

1. Identify which columns have missing values and their percentages
2. For each missing column, determine the missingness type using the correlation method
3. Apply the correct strategy for each column based on the type
4. Train a logistic regression on the cleaned data and report AUC-ROC
5. Add a missingness indicator for any MNAR columns and check if AUC improves

In [ ]:
# Load Titanic
titanic = sns.load_dataset('titanic')

print("Titanic missing values:")
miss = titanic.isnull().sum()
print(miss[miss > 0])
print()
print("Columns:", titanic.columns.tolist())

# YOUR CODE BELOW
# Step 1: Calculate missing % per column


# Step 2: Determine missingness type for 'age' and 'embarked'
# Hint: correlate their missingness indicators with pclass, fare, survived


# Step 3: Apply the right strategy


# Step 4: Encode categoricals, train logistic regression, report AUC


# Step 5: Add indicators where appropriate, check AUC change


In [ ]:
# SOLUTION — run after attempting the exercise above

titanic = sns.load_dataset('titanic')

# Step 1: missing %
print("Missing %:")
print((titanic.isnull().sum() / len(titanic) * 100).round(1)[lambda x: x > 0])

# Step 2: missingness type
age_miss_ind = titanic['age'].isnull().astype(int)
corr_class   = age_miss_ind.corr(titanic['pclass'].astype(float))
corr_fare    = age_miss_ind.corr(titanic['fare'])
corr_surv    = age_miss_ind.corr(titanic['survived'].astype(float))
print(f"\nAge missingness vs pclass: {corr_class:.3f}")
print(f"Age missingness vs fare:   {corr_fare:.3f}")
print(f"Age missingness vs surv:   {corr_surv:.3f}")
print("→ Age is MAR (related to pclass/fare — higher class = less likely to miss age entry)")

# Step 3: apply strategies
t = titanic[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']].copy()

# age: MAR → KNN, add indicator
t['age_was_missing'] = t['age'].isnull().astype(int)
t['age'].fillna(t['age'].median(), inplace=True)

# embarked: 2 rows missing → MCAR → mode imputation
t['embarked'].fillna(t['embarked'].mode()[0], inplace=True)

# encode
t['sex_enc']      = (t['sex'] == 'male').astype(int)
t['embarked_enc'] = t['embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Step 4: baseline model (no indicator)
base_f = ['pclass', 'sex_enc', 'age', 'fare', 'embarked_enc']
X_base = StandardScaler().fit_transform(t[base_f])
y_t    = t['survived']
sc_base = cross_val_score(LogisticRegression(max_iter=300), X_base, y_t, cv=5, scoring='roc_auc')

# Step 5: with indicator
ind_f = base_f + ['age_was_missing']
X_ind = StandardScaler().fit_transform(t[ind_f])
sc_ind = cross_val_score(LogisticRegression(max_iter=300), X_ind, y_t, cv=5, scoring='roc_auc')

print(f"\nBaseline AUC (no indicator): {sc_base.mean():.4f}")
print(f"With age indicator AUC:     {sc_ind.mean():.4f}")
print(f"Difference:                 {(sc_ind.mean() - sc_base.mean())*100:+.2f} AUC points")

## Summary

| Type | Relationship | Safe strategies | Risky strategies |
|------|-------------|-----------------|------------------|
| MCAR | No relation to any variable | Mean, median, drop rows | None |
| MAR  | Related to observed columns | KNN, MICE, median | Drop rows (selection bias) |
| MNAR | Related to missing value itself | Missingness indicator + median | Mean/median alone (systematic bias) |

**The production rule:**  
Fit imputers on training data. Transform test data. Never the reverse.

**The MNAR rule:**  
When data is missing because of its own value, the missingness itself is a feature. Add the indicator column and let the model learn from it.

---

**Day 6 tomorrow: Outliers**  
IQR, Z-score, Isolation Forest — and the domain context that decides whether to remove or keep them.

GitHub repo: https://github.com/VaishnaviJagtap18/42-days-aiml-challenge